<a href="https://colab.research.google.com/github/Parvez089/AI_AND_ML/blob/main/Thesis_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import drive

drive.mount("/content/drive")

drive_dir = "/content/drive/MyDrive/Thesis_Dataset"
extract_root = "/content/extracted_datasets"

datasets = {
    "ds1": os.path.join(drive_dir, "ExDark_DataSet.zip"),
    "ds2": os.path.join(drive_dir, "Flickr.zip"),
    "ds3": os.path.join(drive_dir, "LWFDataset.zip"),
    "ds4": os.path.join(drive_dir, "test2017.zip"),
}

for prefix, zip_path in datasets.items():
  out_dir = os.path.join(extract_root, prefix)
  os.makedirs(out_dir, exist_ok=True)
  if os.path.exists(zip_path):
    print(f"Extracting {os.path.basename(zip_path)}...")
    get_ipython().system(f'unzip -q "{zip_path}" -d "{out_dir}"')
  else:
    print(f"File not found: {zip_path}")

print("All extraction steps completed.")

Mounted at /content/drive
Extracting ExDark_DataSet.zip...
Extracting Flickr.zip...
Extracting LWFDataset.zip...
Extracting test2017.zip...
All extraction steps completed.


In [2]:
import glob
import shutil

raw_merged_dir = "/content/raw_merged_images"
os.makedirs(raw_merged_dir, exist_ok=True)

exts = ("*.jpg", "*.jpeg", "*.png", "*.webp", "*.JPG", "*.JPEG", "*.PNG")
total_merged = 0

for prefix in ["ds1", "ds2", "ds3", "ds4"]:
  src_dir = os.path.join(extract_root, prefix)
  found_files = []
  for ext in exts:
    found_files.extend(
        glob.glob(os.path.join(src_dir, "**", ext), recursive=True)
    )

  # Remove duplicates if case-insensitive matching caught same file twice
  found_files = sorted(set(found_files))

  count = 0
  for fpath in found_files:
    new_name = f"{prefix}_{os.path.basename(fpath)}"
    dst_path = os.path.join(raw_merged_dir, new_name)
    if not os.path.exists(dst_path):
      shutil.copy2(fpath, dst_path)
      count += 1
      total_merged += 1

  print(f"[{prefix}_] merged {count} images.")

print(f"\nTotal merged raw images: {total_merged}")

[ds1_] merged 7363 images.
[ds2_] merged 8091 images.
[ds3_] merged 13233 images.
[ds4_] merged 40670 images.

Total merged raw images: 69357


In [3]:
import cv2
import numpy as np

preprocessed_dir = "/content/Drishti_Merged_320"
os.makedirs(preprocessed_dir, exist_ok=True)


def apply_clahe(img, clip_limit=2.0, tile_grid_size=(8, 8)):
  lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
  l_ch, a_ch, b_ch = cv2.split(lab)
  clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
  cl = clahe.apply(l_ch)
  return cv2.cvtColor(cv2.merge((cl, a_ch, b_ch)), cv2.COLOR_LAB2BGR)


def letterbox_320(img, target_size=(320, 320), pad_color=(114, 114, 114)):
  tw, th = target_size
  h, w = img.shape[:2]
  scale = min(tw / w, th / h)
  nw, nh = int(round(w * scale)), int(round(h * scale))
  resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)

  canvas = np.full((th, tw, 3), pad_color, dtype=np.uint8)
  top = (th - nh) // 2
  left = (tw - nw) // 2
  canvas[top : top + nh, left : left + nw] = resized
  return canvas


image_paths = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
  image_paths.extend(glob.glob(os.path.join(raw_merged_dir, ext)))

processed_count = 0
skipped_count = 0

for img_path in image_paths:
  img = cv2.imread(img_path)
  if img is None:
    skipped_count += 1
    continue

  clahe_applied = apply_clahe(img)
  final_img = letterbox_320(clahe_applied, target_size=(320, 320))

  base_no_ext = os.path.splitext(os.path.basename(img_path))[0]
  out_path = os.path.join(preprocessed_dir, f"{base_no_ext}.jpg")
  cv2.imwrite(out_path, final_img)
  processed_count += 1

print(
    f"Done! Processed: {processed_count}, Skipped (corrupt): {skipped_count}"
)

Done! Processed: 68690, Skipped (corrupt): 0


In [4]:
local_zip_path = "/content/Drishti_Merged_320.zip"
drive_zip_path = (
    "/content/drive/MyDrive/Thesis_Dataset/Drishti_Merged_320.zip"
)

print("Compressing all preprocessed images into a single zip file...")
get_ipython().system(f'zip -rq "{local_zip_path}" "{preprocessed_dir}"')

# Copy final single zip file to Google Drive
shutil.copy2(local_zip_path, drive_zip_path)
print("Saved final archive to Google Drive:", drive_zip_path)
print("Zip size (MB):", round(os.path.getsize(drive_zip_path) / (1024**2), 2))

Compressing all preprocessed images into a single zip file...
Saved final archive to Google Drive: /content/drive/MyDrive/Thesis_Dataset/Drishti_Merged_320.zip
Zip size (MB): 2778.64
